# 👟 Buscador de zapatillas de running al mejor precio

Dado el **modelo** de una zapatilla, el **sexo** y la **talla**, este notebook busca en la web
las tiendas que la venden y devuelve una tabla ordenada de **más barata a más cara**.

**Cómo funciona**

1. **Búsqueda web** (DuckDuckGo, sin API key) con varias consultas para localizar fichas de producto en tiendas online.
2. *(Opcional)* **Google Shopping vía [SerpAPI](https://serpapi.com/)** si defines la variable de entorno `SERPAPI_KEY`: amplía mucho la cobertura.
3. **Análisis de cada ficha de producto**, sin scrapers específicos por tienda:
   - Datos estructurados **schema.org (JSON-LD)** `Product` / `ProductGroup` / `Offer`, que publican la mayoría de e-commerce.
   - Endpoint público **`/products/<handle>.js` de Shopify**, que da precio y stock *por talla*.
   - *Meta tags* de precio (`product:price:amount`, `itemprop="price"`) como último recurso.
4. **Filtrado**: el nombre del producto debe contener el modelo, se descartan los del sexo contrario y se comprueba (cuando la web lo publica) si **tu talla está disponible**.

> ⚖️ Uso personal/educativo. El notebook respeta `robots.txt`, limita el número de peticiones por dominio y no esquiva protecciones anti-bot: las tiendas que bloquean *scrapers* simplemente se omiten.

## 1. Instalación de dependencias

In [ ]:
%pip install -q requests beautifulsoup4 lxml pandas ddgs

## 2. Parámetros de búsqueda

Modifica esta celda y ejecuta el notebook completo (*Run All*).

In [ ]:
MODELO = "Nike Pegasus 41"   # Marca + modelo, p. ej. "ASICS Gel-Nimbus 26", "Adidas Adizero SL 2"
SEXO   = "hombre"            # "hombre", "mujer" o "unisex"
TALLA  = "42.5"              # Talla EU: "42", "42.5", "42 1/2", "42 2/3"...

# --- Ajustes avanzados ---
REGION               = "es-es"   # Región de la búsqueda web (es-es, mx-es, us-en...)
MONEDA               = "EUR"     # Solo se muestran precios en esta moneda (None = todas)
MAX_URLS_BUSQUEDA    = 40        # Nº máximo de páginas de producto a analizar
MAX_URLS_POR_DOMINIO = 3         # Evita saturar una misma tienda
SOLO_TALLA_DISPONIBLE = False    # True = descarta resultados cuya talla no se ha podido confirmar
RESPETAR_ROBOTS      = True      # Consulta robots.txt antes de descargar cada página
TIMEOUT              = 12        # Segundos por petición
HILOS                = 8         # Descargas en paralelo

import os
SERPAPI_KEY = os.getenv("SERPAPI_KEY")  # Opcional: Google Shopping vía SerpAPI

## 3. Imports y utilidades

In [ ]:
import json
import re
import time
import unicodedata
from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import lru_cache
from urllib.parse import urlparse, urlunparse
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import HTML, display

HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                   "(KHTML, like Gecko) Chrome/126.0 Safari/537.36"),
    "Accept-Language": "es-ES,es;q=0.9,en;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/json;q=0.9,*/*;q=0.8",
}

session = requests.Session()
session.headers.update(HEADERS)


def normalizar(texto) -> str:
    """Minúsculas, sin acentos y con cualquier signo convertido en espacio."""
    texto = unicodedata.normalize("NFKD", str(texto or "")).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", " ", texto.lower()).strip()


def dominio(url: str) -> str:
    host = urlparse(url).netloc.lower()
    return host[4:] if host.startswith("www.") else host


def limpiar_url(url: str) -> str:
    """Quita query-string y fragmento para deduplicar."""
    p = urlparse(url)
    return urlunparse((p.scheme, p.netloc, p.path.rstrip("/"), "", "", ""))


def parsear_precio(valor):
    """Convierte '1.299,95 €', '129.99', 12999 (céntimos no) ... en float."""
    if valor is None or isinstance(valor, bool):
        return None
    if isinstance(valor, (int, float)):
        return float(valor) if valor > 0 else None
    s = re.sub(r"[^\d,.]", "", str(valor))
    if not s:
        return None
    if "," in s and "." in s:            # El último separador es el decimal
        if s.rfind(",") > s.rfind("."):
            s = s.replace(".", "").replace(",", ".")
        else:
            s = s.replace(",", "")
    elif "," in s:
        entero, _, dec = s.rpartition(",")
        s = f"{entero.replace(',', '')}.{dec}" if len(dec) <= 2 else s.replace(",", "")
    elif s.count(".") > 1:
        entero, _, dec = s.rpartition(".")
        s = f"{entero.replace('.', '')}.{dec}"
    try:
        precio = float(s)
    except ValueError:
        return None
    return precio if precio > 0 else None

### Tallas, sexo y modelo

Funciones para reconocer la talla en sus distintas notaciones (`42.5`, `42,5`, `42 1/2`, `42½`, `EU 42.5`...),
detectar el sexo en el nombre/URL del producto y comprobar que el producto corresponde al modelo buscado.

In [ ]:
FRACCIONES = {"1/2": ".5", "½": ".5", "1/3": ".33", "⅓": ".33", "2/3": ".66", "⅔": ".66"}


def normalizar_talla(talla) -> str:
    """'42 1/2' -> '42.5', '42,5' -> '42.5', '42 2/3' -> '42.66'."""
    t = str(talla).strip().lower().replace("eu", "").replace(",", ".").strip()
    for frac, dec in FRACCIONES.items():
        t = re.sub(rf"\s*{re.escape(frac)}", dec, t)
    return t


def patron_talla(talla) -> re.Pattern:
    """Regex que reconoce la talla dentro de un texto sin confundir 42 con 42.5 o 142."""
    t = normalizar_talla(talla)
    entero, _, dec = t.partition(".")
    no_sigue = r"(?![\d]|[.,]\d|\s?\d/\d|[½⅓⅔])"
    if not dec:
        cuerpo = re.escape(entero)
    elif dec == "5":
        cuerpo = rf"{entero}(?:[.,]5|\s?1/2|\s?½)"
    elif dec.startswith("3"):
        cuerpo = rf"{entero}(?:[.,]33?|\s?1/3|\s?⅓)"
    elif dec.startswith("6"):
        cuerpo = rf"{entero}(?:[.,]66?|[.,]67|\s?2/3|\s?⅔)"
    else:
        cuerpo = rf"{entero}[.,]{dec}"
    return re.compile(rf"(?<![\d.,]){cuerpo}{no_sigue}", re.IGNORECASE)


PALABRAS_SEXO = {
    "hombre": {"hombre", "hombres", "men", "mens", "man", "masculino", "caballero", "homme", "uomo", "herren"},
    "mujer":  {"mujer", "mujeres", "women", "womens", "woman", "wmns", "femenino", "senora", "femme", "donna", "damen"},
}


def sexo_detectado(texto: str) -> set:
    palabras = set(normalizar(texto).split())
    return {sexo for sexo, claves in PALABRAS_SEXO.items() if palabras & claves}


def sexo_compatible(texto: str, sexo: str) -> bool:
    """Descarta productos marcados solo con el sexo contrario. Unisex / sin marcar -> se acepta."""
    sexo = normalizar(sexo)
    if sexo not in PALABRAS_SEXO:
        return True
    encontrados = sexo_detectado(texto)
    return not encontrados or sexo in encontrados or "unisex" in normalizar(texto).split()


def coincide_modelo(texto: str, modelo: str) -> bool:
    """Todas las palabras del modelo deben aparecer (como palabras completas) en el texto."""
    palabras = set(normalizar(texto).split())
    return all(tok in palabras for tok in normalizar(modelo).split())

## 4. Búsqueda de páginas de producto en la web

In [ ]:
DOMINIOS_EXCLUIDOS = {
    "youtube.com", "facebook.com", "instagram.com", "tiktok.com", "twitter.com", "x.com",
    "pinterest.com", "pinterest.es", "reddit.com", "wikipedia.org", "strava.com",
    "runrepeat.com", "believeintherun.com", "doctorsofrunning.com", "roadtrailrun.com",
    "google.com", "bing.com", "duckduckgo.com",
}

RUTAS_NO_PRODUCTO = re.compile(r"/(blog|noticias|news|review|reviews|analisis|foro|forum|comparativa)/", re.I)


def es_url_candidata(url: str) -> bool:
    if not url.startswith("http"):
        return False
    d = dominio(url)
    if any(d == ex or d.endswith("." + ex) for ex in DOMINIOS_EXCLUIDOS):
        return False
    return not RUTAS_NO_PRODUCTO.search(urlparse(url).path)


def consultas_busqueda(modelo, sexo, talla):
    s = "" if normalizar(sexo) == "unisex" else sexo
    return [
        f"{modelo} {s} zapatillas comprar precio",
        f"{modelo} {s} talla {talla} oferta",
        f"zapatillas running {modelo} {s} tienda",
        f"{modelo} {s} running shoes price",
    ]


def buscar_urls_ddg(modelo, sexo, talla, max_por_consulta=25):
    from ddgs import DDGS
    urls = []
    with DDGS() as ddgs:
        for q in consultas_busqueda(modelo, sexo, talla):
            try:
                resultados = ddgs.text(q, region=REGION, safesearch="off", max_results=max_por_consulta)
            except Exception as e:
                print(f"  ⚠️ Búsqueda fallida '{q}': {e}")
                continue
            for r in resultados or []:
                urls.append({"url": r.get("href", ""), "titulo": r.get("title", ""), "fuente": "web"})
            time.sleep(1)  # Cortesía con el buscador
    return urls


def buscar_google_shopping(modelo, sexo, talla):
    """Resultados de Google Shopping vía SerpAPI (requiere SERPAPI_KEY)."""
    if not SERPAPI_KEY:
        return []
    gl, _, hl = REGION.partition("-")   # "es-es" -> país "es", idioma "es"
    params = {"engine": "google_shopping", "q": f"{modelo} {sexo} zapatillas",
              "gl": gl, "hl": hl or gl, "api_key": SERPAPI_KEY, "num": 100}
    try:
        data = session.get("https://serpapi.com/search.json", params=params, timeout=30).json()
    except Exception as e:
        print(f"  ⚠️ SerpAPI falló: {e}")
        return []
    ofertas = []
    for r in data.get("shopping_results", []):
        ofertas.append({
            "tienda": r.get("source"),
            "producto": r.get("title"),
            "precio": r.get("extracted_price") or parsear_precio(r.get("price")),
            "moneda": MONEDA or "",
            "talla_disponible": "?",
            "url": r.get("link") or r.get("product_link"),
            "fuente": "google-shopping",
        })
    return ofertas


def seleccionar_urls(candidatas):
    """Filtra, deduplica y limita el nº de URLs por dominio."""
    vistas, por_dominio, elegidas = set(), {}, []
    for c in candidatas:
        url = c["url"]
        if not es_url_candidata(url):
            continue
        clave = limpiar_url(url)
        d = dominio(url)
        if clave in vistas or por_dominio.get(d, 0) >= MAX_URLS_POR_DOMINIO:
            continue
        vistas.add(clave)
        por_dominio[d] = por_dominio.get(d, 0) + 1
        elegidas.append(c)
        if len(elegidas) >= MAX_URLS_BUSQUEDA:
            break
    return elegidas

## 5. Extracción de precio y tallas de cada ficha de producto

In [ ]:
@lru_cache(maxsize=None)
def robots_para(host_base: str):
    rp = RobotFileParser()
    try:
        r = session.get(f"{host_base}/robots.txt", timeout=TIMEOUT)
        rp.parse(r.text.splitlines() if r.status_code == 200 else [])
    except requests.RequestException:
        rp.parse([])
    return rp


def permitido(url: str) -> bool:
    if not RESPETAR_ROBOTS:
        return True
    p = urlparse(url)
    return robots_para(f"{p.scheme}://{p.netloc}").can_fetch(HEADERS["User-Agent"], url)


def descargar(url: str):
    if not permitido(url):
        return None
    try:
        r = session.get(url, timeout=TIMEOUT)
    except requests.RequestException:
        return None
    return r if r.status_code == 200 else None


# ---------- schema.org JSON-LD ----------
def _iterar_nodos(obj):
    if isinstance(obj, list):
        for x in obj:
            yield from _iterar_nodos(x)
    elif isinstance(obj, dict):
        yield obj
        for clave in ("@graph", "mainEntity", "itemListElement", "item"):
            if clave in obj:
                yield from _iterar_nodos(obj[clave])


def _tipos(nodo):
    t = nodo.get("@type", [])
    return {str(x).split("/")[-1].lower() for x in (t if isinstance(t, list) else [t])}


def _como_lista(x):
    return x if isinstance(x, list) else ([x] if x else [])


def _precio_oferta(oferta):
    precio = oferta.get("price") or oferta.get("lowPrice")
    spec = oferta.get("priceSpecification")
    if precio is None and spec:
        precio = next((s.get("price") for s in _como_lista(spec) if isinstance(s, dict) and s.get("price")), None)
    moneda = oferta.get("priceCurrency") or next(
        (s.get("priceCurrency") for s in _como_lista(spec) if isinstance(s, dict)), None)
    return parsear_precio(precio), moneda


def _disponible(oferta):
    disp = str(oferta.get("availability", "")).lower()
    if not disp:
        return None
    return any(k in disp for k in ("instock", "limitedavailability", "onlineonly", "presale", "preorder"))


def extraer_jsonld(soup):
    """Devuelve (nombre, [ {precio, moneda, disponible, texto_talla} ])."""
    nombre, variantes = None, []
    for script in soup.find_all("script", type=re.compile("ld\\+json", re.I)):
        try:
            data = json.loads(script.string or script.get_text() or "{}", strict=False)
        except (json.JSONDecodeError, TypeError):
            continue
        for nodo in _iterar_nodos(data):
            tipos = _tipos(nodo)
            if not tipos & {"product", "productgroup"}:
                continue
            nombre = nombre or nodo.get("name")
            variantes_prod = [v for v in _como_lista(nodo.get("hasVariant")) if isinstance(v, dict)]
            for prod in [nodo] + variantes_prod:
                es_variante = prod is not nodo
                talla_campo = str(prod.get("size", "") or "")
                texto_prod = " ".join(str(prod.get(k, "")) for k in ("name", "sku")) if es_variante else ""
                for oferta in _como_lista(prod.get("offers")):
                    if not isinstance(oferta, dict):
                        continue
                    for o in _como_lista(oferta.get("offers")) or [oferta]:   # AggregateOffer
                        if not isinstance(o, dict):
                            continue
                        precio, moneda = _precio_oferta(o)
                        if precio is None:
                            precio, moneda = _precio_oferta(oferta)
                        if precio is None:
                            continue
                        texto_oferta = " ".join(str(o.get(k, "")) for k in ("name", "sku", "size"))
                        variantes.append({
                            "precio": precio,
                            "moneda": moneda,
                            "disponible": _disponible(o),
                            "texto_talla": f"{talla_campo} {texto_prod} {texto_oferta}".strip(),
                            "talla_explicita": bool(talla_campo or o.get("size")),
                        })
    return nombre, variantes


# ---------- Shopify ----------
def extraer_shopify(url, html):
    if "shopify" not in html.lower() or "/products/" not in url:
        return None, []
    base = limpiar_url(url)
    handle_url = base[: base.index("/products/")] + "/products/" + base.split("/products/")[1].split("/")[0]
    r = descargar(handle_url + ".js")
    if r is None:
        return None, []
    try:
        data = r.json()
    except ValueError:
        return None, []
    opciones = [str(o.get("name", "") if isinstance(o, dict) else o).lower() for o in data.get("options", [])]
    idx_talla = [i for i, o in enumerate(opciones)
                 if any(k in o for k in ("talla", "size", "taglia", "taille", "tamano", "tamaño", "numero"))]
    m = re.search(r'Shopify\.currency\s*=\s*\{"active":"(\w{3})"', html)
    moneda = m.group(1) if m else None
    variantes = []
    for v in data.get("variants", []):
        valores = [v.get(f"option{i + 1}") or "" for i in range(3)]
        texto = " ".join(valores[i] for i in idx_talla) if idx_talla else v.get("title", "")
        precio = v.get("price")
        variantes.append({
            "precio": precio / 100 if isinstance(precio, (int, float)) else parsear_precio(precio),
            "moneda": moneda,
            "disponible": v.get("available"),
            "texto_talla": texto,
            "talla_explicita": bool(idx_talla),
        })
    return data.get("title"), variantes


# ---------- Meta tags (último recurso) ----------
def extraer_meta(soup):
    precio = moneda = None
    for prop in ("product:price:amount", "og:price:amount"):
        tag = soup.find("meta", attrs={"property": prop})
        if tag and tag.get("content"):
            precio = parsear_precio(tag["content"])
            cur = soup.find("meta", attrs={"property": prop.replace("amount", "currency")})
            moneda = cur.get("content") if cur else None
            break
    if precio is None:
        tag = soup.find(attrs={"itemprop": "price"})
        if tag:
            precio = parsear_precio(tag.get("content") or tag.get_text())
            cur = soup.find(attrs={"itemprop": "priceCurrency"})
            moneda = (cur.get("content") or cur.get_text()) if cur else None
    return ([{"precio": precio, "moneda": moneda, "disponible": None, "texto_talla": "", "talla_explicita": False}]
            if precio else [])


def nombre_pagina(soup):
    for attrs in ({"property": "og:title"}, {"name": "twitter:title"}):
        tag = soup.find("meta", attrs=attrs)
        if tag and tag.get("content"):
            return tag["content"]
    h1 = soup.find("h1")
    if h1:
        return h1.get_text(" ", strip=True)
    return soup.title.get_text(strip=True) if soup.title else ""


def estado_talla(variantes, talla, modelo):
    """Busca la talla entre las variantes. Devuelve ('Sí'|'No'|'?', variante_más_barata)."""
    patron = patron_talla(talla)
    # Quita el nombre del modelo del texto: en "Pegasus 41 - 42.5" el 41 no es una talla
    patron_modelo = re.compile(r"[\W_]*".join(map(re.escape, normalizar(modelo).split())), re.I)
    for v in variantes:
        v["texto_talla"] = patron_modelo.sub(" ", v["texto_talla"])

    de_la_talla = [v for v in variantes if patron.search(v["texto_talla"])]
    if de_la_talla:
        disponibles = [v for v in de_la_talla if v["disponible"] is not False]
        if disponibles:
            mejor = min(disponibles, key=lambda v: v["precio"])
            return ("Sí" if mejor["disponible"] else "?"), mejor
        return "No", min(de_la_talla, key=lambda v: v["precio"])
    # La web publica sus tallas y la nuestra no está entre ellas -> no la vende
    if sum(1 for v in variantes if v["talla_explicita"]) >= 3:
        return "No", min(variantes, key=lambda v: v["precio"])
    en_stock = [v for v in variantes if v["disponible"] is not False] or variantes
    return "?", min(en_stock, key=lambda v: v["precio"])


def analizar_pagina(candidata, modelo, sexo, talla):
    url = candidata["url"]
    r = descargar(url)
    if r is None or "html" not in r.headers.get("Content-Type", "html"):
        return None
    html = r.text
    soup = BeautifulSoup(html, "lxml")

    nombre, variantes = extraer_shopify(r.url, html)
    if not variantes:
        nombre, variantes = extraer_jsonld(soup)
    if not variantes:
        variantes = extraer_meta(soup)
    variantes = [v for v in variantes if v["precio"]]
    if not variantes:
        return None

    nombre = nombre or nombre_pagina(soup) or candidata.get("titulo", "")
    texto_ident = f"{nombre} {urlparse(r.url).path} {candidata.get('titulo', '')}"
    if not coincide_modelo(f"{nombre} {urlparse(r.url).path}", modelo):
        return None
    if not sexo_compatible(texto_ident, sexo):
        return None

    talla_ok, variante = estado_talla(variantes, talla, modelo)
    moneda = variante["moneda"] or next((v["moneda"] for v in variantes if v["moneda"]), None)
    if moneda is None and "€" in html[:200_000]:
        moneda = "EUR"
    return {
        "tienda": dominio(r.url),
        "producto": re.sub(r"\s+", " ", str(nombre)).strip(),
        "precio": round(variante["precio"], 2),
        "moneda": (moneda or "").upper(),
        "talla_disponible": talla_ok,
        "url": r.url,
        "fuente": candidata.get("fuente", "web"),
    }

## 6. Función principal

In [ ]:
def buscar_zapatillas(modelo: str, sexo: str, talla, verbose: bool = True) -> pd.DataFrame:
    t0 = time.time()
    log = print if verbose else (lambda *a, **k: None)

    log(f"🔎 Buscando '{modelo}' · {sexo} · talla {talla} ...")
    candidatas = buscar_urls_ddg(modelo, sexo, talla)
    ofertas = buscar_google_shopping(modelo, sexo, talla)
    log(f"   {len(candidatas)} resultados web, {len(ofertas)} de Google Shopping")

    # Las URLs directas de Google Shopping también se analizan para comprobar la talla
    for o in ofertas:
        if o["url"] and "google." not in dominio(o["url"]):
            candidatas.append({"url": o["url"], "titulo": o["producto"], "fuente": "google-shopping"})

    urls = seleccionar_urls(candidatas)
    log(f"   Analizando {len(urls)} páginas de producto ...")

    resultados = []
    with ThreadPoolExecutor(max_workers=HILOS) as pool:
        futuros = {pool.submit(analizar_pagina, c, modelo, sexo, talla): c for c in urls}
        for f in as_completed(futuros):
            try:
                res = f.result()
            except Exception as e:  # Una tienda rota no debe romper la búsqueda
                log(f"   ⚠️ {futuros[f]['url']}: {e}")
                continue
            if res:
                resultados.append(res)

    # Ofertas de Google Shopping que no se pudieron verificar en la propia tienda
    analizadas = " ".join(r["tienda"].replace(".", "") for r in resultados)
    for o in ofertas:
        ya_analizada = normalizar(o["tienda"]).replace(" ", "") in analizadas
        if (o["precio"] and coincide_modelo(o["producto"], modelo)
                and sexo_compatible(o["producto"], sexo) and not ya_analizada):
            resultados.append(o)

    df = pd.DataFrame(resultados, columns=["tienda", "producto", "precio", "moneda",
                                           "talla_disponible", "url", "fuente"])
    if df.empty:
        log("❌ No se han encontrado resultados.")
        return df

    if MONEDA:
        df = df[(df["moneda"] == MONEDA.upper()) | (df["moneda"] == "")]
    if SOLO_TALLA_DISPONIBLE:
        df = df[df["talla_disponible"] == "Sí"]
    else:
        df = df[df["talla_disponible"] != "No"]

    df = (df.sort_values("precio")
            .drop_duplicates(subset=["tienda"], keep="first")   # la oferta más barata de cada tienda
            .reset_index(drop=True))
    df.index += 1
    log(f"✅ {len(df)} tiendas con precio en {time.time() - t0:.0f}s")
    return df

## 7. ¡Buscar!

In [ ]:
df = buscar_zapatillas(MODELO, SEXO, TALLA)

def mostrar(df, n=20):
    if df.empty:
        return
    tabla = df.head(n).copy()
    tabla["precio"] = tabla.apply(lambda r: f"{r.precio:,.2f} {r.moneda}".strip(), axis=1)
    tabla["url"] = tabla["url"].apply(lambda u: f'<a href="{u}" target="_blank">Ver oferta</a>')
    tabla["talla_disponible"] = tabla["talla_disponible"].map({"Sí": "✅ Sí", "?": "❔ Sin confirmar"})
    display(HTML(tabla.drop(columns=["moneda"]).to_html(escape=False)))

mostrar(df)

**Leyenda de `talla_disponible`**

- ✅ **Sí**: la tienda publica stock por talla y la tuya está disponible.
- ❔ **Sin confirmar**: la tienda no expone las tallas de forma legible (suelen cargarse con JavaScript); abre el enlace para comprobarlo.
- Los resultados en los que la talla consta como **agotada o inexistente se descartan** automáticamente.

## 8. La opción más barata y exportación

In [ ]:
if not df.empty:
    mejor = df.iloc[0]
    print(f"🏆 Más barata: {mejor.producto}\n   {mejor.precio:.2f} {mejor.moneda} en {mejor.tienda}"
          f" (talla {TALLA}: {mejor.talla_disponible})\n   {mejor.url}")

    os.makedirs("../data", exist_ok=True)
    nombre_csv = f"../data/{normalizar(MODELO).replace(' ', '_')}_{normalizar(SEXO)}_{normalizar_talla(TALLA)}.csv"
    df.to_csv(nombre_csv, index=False, encoding="utf-8-sig")
    print(f"\n💾 Resultados guardados en {nombre_csv}")

## 💡 Consejos

- **Sé específico con el modelo**: incluye marca y versión (`"Nike Pegasus 41"` mejor que `"Pegasus"`). Todas las palabras del modelo deben aparecer en el nombre del producto, así se evita mezclar versiones (41 vs 40).
- **Más cobertura**: crea una cuenta gratuita en [SerpAPI](https://serpapi.com/) y define `SERPAPI_KEY` para añadir los resultados de Google Shopping.
- **Muchos "❔ Sin confirmar"**: pon `SOLO_TALLA_DISPONIBLE = True` para quedarte solo con las tiendas donde se ha verificado la talla.
- Si una búsqueda devuelve pocos resultados, sube `MAX_URLS_BUSQUEDA` o prueba otra `REGION`.